# Análisis exploratorio

Seminario Final · Lic. en Ciencia de Datos, UNaB · Facundo Tarizzo

## Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2_contingency

AZUL    = '#1f4e79'
TEAL    = '#2a6f6f'
CELESTE = '#5b8bbf'
NEUTRO  = '#8a97a3'
TEXTO   = '#2d3748'

SEQ = LinearSegmentedColormap.from_list(
    'seq', ['#ffffff', '#dae3f0', '#a3bdd9', '#5b8bbf', '#1f4e79', '#12314a'])
DIV = LinearSegmentedColormap.from_list(
    'div', ['#1f4e79', '#8faed0', '#f7f7f7', '#c99aa3', '#7b2d3b'])

mpl.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelcolor': TEXTO,
    'axes.edgecolor': '#c8cdd3',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#eceff2',
    'grid.linewidth': 0.7,
    'xtick.color': TEXTO,
    'ytick.color': TEXTO,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.frameon': False,
    'legend.fontsize': 9,
})

In [ ]:
conn = sqlite3.connect('/content/drive/MyDrive/Tesis/FPA_FOD_20170508.sqlite')
df = pd.read_sql("SELECT * FROM Fires;", conn)
conn.close()

cols = ['FOD_ID', 'FIRE_YEAR', 'DISCOVERY_DOY', 'DISCOVERY_TIME', 'STAT_CAUSE_DESCR',
        'CONT_DOY', 'FIRE_SIZE', 'FIRE_SIZE_CLASS', 'LATITUDE', 'LONGITUDE',
        'OWNER_DESCR', 'STATE', 'COUNTY', 'NWCG_REPORTING_AGENCY',
        'NWCG_REPORTING_UNIT_NAME', 'SOURCE_SYSTEM_TYPE']
df = df[cols].copy()

df['SIN_CAUSA'] = (df['STAT_CAUSE_DESCR'] == 'Missing/Undefined').astype(int)
df['TIPO'] = np.where(df['SIN_CAUSA']==1, 'Sin causa',
             np.where(df['STAT_CAUSE_DESCR']=='Lightning', 'Natural', 'Antrópica'))
df['MES'] = pd.to_datetime(df['DISCOVERY_DOY'], format='%j').dt.month

CAUSAS = df['STAT_CAUSE_DESCR'].value_counts().index.tolist()

OCRE = '#d4a017'

def color_causa(c):
    if c == 'Lightning':         return OCRE
    if c == 'Missing/Undefined': return NEUTRO
    return AZUL

print(df.shape)
print(df['TIPO'].value_counts().to_string())

### Limpieza

De las 39 columnas originales me quedo con 16. Descarto las que tienen más del 50% de faltantes (`MTBS_*`, `ICS_209_*`, `COMPLEX_NAME`, `FIRE_CODE`, `LOCAL_*`, `FIRE_NAME`), Estas 16 columnas son las que uso para explorar. Varias no van a entrar al modelo: `STAT_CAUSE_DESCR` es de donde sale la variable objetivo, `SOURCE_SYSTEM_TYPE` sirve para entender el sub-reporte pero no para predecir causa, `FIRE_SIZE_CLASS` es una versión discretizada de `FIRE_SIZE`, y `DISCOVERY_TIME`, `CONT_DOY` y `COUNTY` tienen demasiados faltantes. Los terminan usando ocho.

Antes de seguir verifico que no haya duplicados, coordenadas fuera de rango ni tamaños
negativos.

In [ ]:
print("Duplicados:", df.duplicated().sum())
print("Coordenadas fuera de rango:",
      ((df['LATITUDE'].abs()>90) | (df['LONGITUDE'].abs()>180)).sum())
print("Tamaños negativos o cero:", (df['FIRE_SIZE'] <= 0).sum())
print()
print("Faltantes por columna (%)")
print((df.isna().mean()*100).round(2).sort_values(ascending=False).to_string())

## Distribución de causas

In [ ]:
conteo = df['STAT_CAUSE_DESCR'].value_counts()
pct = conteo / len(df) * 100

fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.barh(range(len(conteo)), conteo.values,
        color=[color_causa(c) for c in conteo.index], height=0.72)
ax.set_yticks(range(len(conteo)))
ax.set_yticklabels(conteo.index)
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x/1000)}k'))
ax.set_xlim(0, conteo.max()*1.30)
ax.set_xlabel('Cantidad de incendios')
ax.grid(axis='y', visible=False)

for i, (v, p) in enumerate(zip(conteo.values, pct.values)):
    ax.text(v + conteo.max()*0.015, i, f'{v:,}   ({p:.1f}%)',
            va='center', fontsize=8.5, color=TEXTO)

handles = [plt.Rectangle((0,0),1,1, color=x) for x in [TEAL, AZUL, NEUTRO]]
ax.legend(handles, ['Natural (rayo)', 'Antrópica', 'Sin causa reportada'], loc='lower right')
ax.set_title('Distribución de causas de ignición', loc='left', pad=12)
fig.text(0.005, -0.02, 'FPA-FOD, 1992–2015 · n = 1.880.465', fontsize=8.5, color='#8a97a3')
plt.show()
plt.savefig('../figuras/causas.png')

Las tres causas más frecuentes son humanas y ninguna llega al 25%: quema de
residuos, causas varias e incendios intencionales. Las tormentas electricas, que es la única causa natural del dataset, queda cuarto con 278.468 casos.

Los registros sin causa (166.723) aparecen más que ocho de las doce causas conocidas.

## Estacionalidad de cada causa

In [ ]:
nombres_mes = ['E','F','M','A','M','J','J','A','S','O','N','D']

fig, axes = plt.subplots(4, 4, figsize=(16.5, 11))

for ax, causa in zip(axes.flat, CAUSAS):
    sub = df[df['STAT_CAUSE_DESCR']==causa]
    m = sub['MES'].value_counts().reindex(range(1,13), fill_value=0)

    ax.bar(range(1,13), m.values, color=color_causa(causa), width=0.75)
    ax.set_title(causa, fontsize=10, loc='left')
    ax.set_xticks(range(1,13))
    ax.set_xticklabels(nombres_mes, fontsize=7.5)
    ax.set_ylim(0, m.max()*1.10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v, p: f'{int(v/1000)}k' if v >= 1000 else f'{int(v)}'))
    ax.tick_params(axis='y', labelsize=7.5)
    ax.grid(axis='x', visible=False)

for ax in axes.flat[len(CAUSAS):]:
    ax.axis('off')

fig.suptitle('Estacionalidad de cada causa — incendios por mes',
             x=0.005, ha='left', fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()
plt.savefig('../figuras/estacionalidad.png')

La causa natural se concentra de forma muy marcada en julio y agosto, que son los meses de tormentas eléctricas de verano en Estados Unidos. Fuera de la ventana junio-septiembre casi no aparece.

Las causas humanas tienen otra forma: quema de residuos, incendios intencionales y fuegos iniciados por menores tienen su pico en marzo y abril, al final del invierno, cuando se limpian terrenos y la vegetación seca todavía no rebrotó. Otras, como el uso de equipos o el tabaquismo, se reparten bastante parejo todo el año.

Cada panel tiene su propia escala en el eje vertical, así se compara la forma de cada causa y no el volumen.

## Evolución por década

In [ ]:
def decada(a):
    if a <= 1999: return '1992-1999'
    if a <= 2009: return '2000-2009'
    return '2010-2015'

df['DECADA'] = df['FIRE_YEAR'].apply(decada)
orden_dec = ['1992-1999','2000-2009','2010-2015']
tab = df.groupby(['DECADA','TIPO']).size().unstack(fill_value=0).loc[orden_dec]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (tipo, color) in zip(axes, [('Antrópica', AZUL), ('Natural', OCRE), ('Sin causa', NEUTRO)]):
    vals = tab[tipo]
    ax.bar(range(3), vals.values, color=color, width=0.6)
    for i, v in enumerate(vals.values):
        ax.text(i, v + vals.max()*0.035, f'{v:,}', ha='center',
                fontsize=10, fontweight='bold', color=TEXTO)
    ax.set_xticks(range(3)); ax.set_xticklabels(orden_dec, fontsize=9)
    ax.set_ylim(0, vals.max()*1.20)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,p: f'{int(v/1000)}k'))
    ax.set_title(tipo, loc='left', color=color)
    ax.grid(axis='x', visible=False)

fig.suptitle('Incendios por década según tipo de causa',
             x=0.005, ha='left', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Composición relativa (%)")
print((tab.div(tab.sum(axis=1), axis=0)*100).round(1).to_string())
plt.savefig('../figuras/decadas.png')

Lo que importa es que la composición relativa se mantiene estable: los antrópicos rondan el 76-78% en los tres períodos y los naturales entre 13% y 15%.  La proporción de registros sin causa también se mantiene cerca del 9% en los tres períodos.

## Registros sin causa por estado y año

In [ ]:
tot = df.groupby(['STATE','FIRE_YEAR']).size().unstack(fill_value=0)
mis = df[df['SIN_CAUSA']==1].groupby(['STATE','FIRE_YEAR']).size().unstack(fill_value=0)
mis = mis.reindex(index=tot.index, columns=tot.columns, fill_value=0)
pct_mis = (mis / tot.replace(0, np.nan) * 100)

top20 = df[df['SIN_CAUSA']==1]['STATE'].value_counts().head(20).index
pct_mis = pct_mis.loc[pct_mis.index.intersection(top20)]
pct_mis = pct_mis.loc[pct_mis.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(15, 8.5))
sns.heatmap(pct_mis, cmap=SEQ, vmin=0, vmax=100, linewidths=1.2, linecolor='white',
            cbar_kws={'label': '% de incendios sin causa reportada', 'shrink': 0.55, 'pad': 0.015},
            ax=ax)
ax.set_title('Registros sin causa por estado y año 20 estados con más missing',
             loc='left', pad=14)
ax.set_xlabel(''); ax.set_ylabel('')
ax.tick_params(axis='y', rotation=0, labelsize=10, length=0)
ax.tick_params(axis='x', rotation=0, labelsize=9, length=0)
ax.grid(visible=False)
for s in ax.spines.values():
    s.set_visible(False)
plt.tight_layout()
plt.show()
plt.savefig('../figuras/subreporte_heatmap.png')

In [ ]:
pl = pct_mis.stack().reset_index()
pl.columns = ['STATE','FIRE_YEAR','PCT']
bloques = pl[pl['PCT'] > 80]
en_bloques = df[df['SIN_CAUSA']==1].merge(bloques[['STATE','FIRE_YEAR']], on=['STATE','FIRE_YEAR'])

print(f"Combinaciones estado-año con más del 80% sin causa: {len(bloques)}")
print(f"Concentran el {len(en_bloques)/df['SIN_CAUSA'].sum()*100:.1f}% del total de registros sin causa")

Las celdas en blanco pueden significar dos cosas distintas: que el estado reportó bien o
que casi no hubo incendios registrados ese año. Para distinguirlas, el gráfico siguiente
muestra el volumen junto con la proporción sin causa.

In [ ]:
CASOS = [
    ('HI', 'Crónico: casi nunca se registra la causa'),
    ('SC', 'Cambio de criterio: se empieza a registrar a partir de cierto año'),
    ('NC', 'Episodio acotado: falla concentrada en pocos años'),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 11.5))
anios = tot.columns.values

for ax, (est, subtitulo) in zip(axes, CASOS):
    con = (tot.loc[est] - mis.loc[est]).values
    sin = mis.loc[est].values
    total = con + sin

    ax.bar(anios, con, color='#c9d6e2', width=0.78, label='Con causa registrada')
    ax.bar(anios, sin, bottom=con, color=AZUL, width=0.78, label='Sin causa')

    for x, c, s, t in zip(anios, con, sin, total):
        if t > 0 and s / t > 0.5:
            ax.text(x, t + total.max()*0.045, f'{s/t*100:.0f}%',
                    ha='center', fontsize=7.5, color=AZUL, fontweight='bold')

    ax.set_title(f'{est}  —  {subtitulo}', loc='left', fontsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda v,p: f'{int(v/1000)}k' if v >= 1000 else f'{int(v)}'))
    ax.set_ylim(0, total.max()*1.20)
    ax.set_ylabel('Incendios')
    ax.set_xlim(1991.3, 2015.7)
    ax.set_xticks(anios)
    ax.set_xticklabels(anios, rotation=90, fontsize=7.5)
    ax.grid(axis='x', visible=False)

axes[0].legend(loc='upper left')
axes[-1].set_xlabel('Año')

fig.suptitle('Tres formas distintas de sub-reporte',
             x=0.005, ha='left', fontsize=14, fontweight='bold')
fig.text(0.005, -0.015,
         'La altura total de la barra es la cantidad de incendios registrados ese año. La parte oscura es la porción sin causa.\n'
         'Así se distingue un año sin registros (barra baja) de un año con muchos incendios mal documentados (barra alta y oscura).',
         fontsize=9, color='#8a97a3')
plt.tight_layout()
plt.show()
plt.savefig('../figuras/subreporte_casos.png')

El sub-reporte no se reparte parejo: hay 62 combinaciones de estado y año
donde más del 80% de los incendios no tiene causa anotada, y esas 62 concentran el 60,6% de todos los registros sin causa del dataset.

Bajo mi punto de vista se puede distinguir tres formas de sub-reporte. Hay estados crónicos, que casi nunca registran la causa en todo el período (Hawaii, Puerto Rico). Hay estados con un cambio de criterio, donde a partir de cierto año empiezan a completar el campo (Carolina del Sur, Maine, Oklahoma). Y hay episodios acotados, donde la falla se concentra en pocos años y el resto del tiempo el reporte es normal (Carolina del Norte, Misisipi, Iowa).

Revisando `SOURCE_SYSTEM_TYPE` aparece el mecanismo detrás de varios de estos casos: el sub-reporte no sigue al incendio sino al canal por el que entró el registro. En Hawaii el 100% de los casos sin causa proviene de registros interagenciales, mientras que los federales y los estatales tienen 0%; además ese canal aparece recién en 2001 y desaparece en 2014.
En Carolina del Norte pasa algo parecido (88,5% contra 15,2%). Carolina del Sur es la excepción: tiene un solo registro interagencial en 24 años, así que ahí la fuente no cambió y el problema está dentro del propio sistema estatal. Coincide con lo que declara la Comisión Forestal de Carolina del Sur, que reconoce no contar con un sistema uniforme de reporte.

La documentación oficial del FPA-FOD aclara que no hubo registros estatales y locales disponibles para todos los estados en todos los años. Y en ediciones posteriores del dataset varios de estos huecos ya fueron completados con datos.

### Sub-reporte según la agencia que reporta

In [ ]:
ag = pd.crosstab(df['NWCG_REPORTING_AGENCY'], df['SIN_CAUSA']==1, normalize='index')[True] * 100
tabla_ag = pd.DataFrame({
    'Agencia': ag.index,
    'Registros': df['NWCG_REPORTING_AGENCY'].value_counts()[ag.index].values,
    '% sin causa': ag.values.round(2)
}).sort_values('Registros', ascending=False)
print(tabla_ag.to_string(index=False))

### Sub-reporte según la agencia que reporta

| Agencia | Qué es | Registros | % sin causa |
|---|---|---|---|
| ST/C&L | Estado, condado o local | 1.377.090 | 9,92% |
| FS | Forest Service | 220.497 | 0,03% |
| BIA | Bureau of Indian Affairs | 119.943 | 0,40% |
| BLM | Bureau of Land Management | 97.034 | 5,86% |
| IA | Organización interagencial | 21.841 | 99,88% |
| NPS | National Park Service | 20.893 | 5,50% |
| FWS | Fish & Wildlife Service | 19.331 | 4,61% |
| TRIBE | Agencias tribales | 3.739 | 0,32% |
| DOD | Departamento de Defensa | 81 | 90,12% |
| BOR | Bureau of Reclamation | 14 | 21,43% |
| DOE | Departamento de Energía | 2 | 100,00% |


## Causa natural según el propietario del terreno

In [ ]:
dfc = df[df['SIN_CAUSA']==0]

own = pd.crosstab(dfc['OWNER_DESCR'], dfc['TIPO'], normalize='index') * 100
own['n'] = dfc['OWNER_DESCR'].value_counts()
own = own[own['n'] > 1000].sort_values('Natural')

fig, ax = plt.subplots(figsize=(11, 6.5))
y = np.arange(len(own))

ax.barh(y, own['Natural'], color=OCRE, height=0.68, label='Natural (rayo)')
ax.barh(y, own['Antrópica'], left=own['Natural'], color='#c9d6e2', height=0.68, label='Antrópica')

for i, nat in enumerate(own['Natural']):
    ax.text(nat/2, i, f'{nat:.0f}%', ha='center', va='center',
            fontsize=9.5, fontweight='bold', color='white')

ax.set_yticks(y)
ax.set_yticklabels([f'{i}   ({n:,})' for i, n in zip(own.index, own['n'])], fontsize=9.5)
ax.set_xlim(0, 100)
ax.set_xlabel('% de los incendios de ese tipo de terreno')
ax.set_title('Peso de la causa natural según el propietario del terreno', loc='left', pad=12)
ax.legend(loc='lower right')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()
plt.savefig('../figuras/propietario.png')

En tierras del Bureau of Land Management el 71% de los incendios son por
rayo, y en el Forest Service el 60%. En terrenos privados esa proporción cae al 9%.

Tiene sentido: las tierras federales están en zonas remotas del oeste, con poca actividad humana y mucha tormenta seca. Los terrenos privados están donde vive la gente.

La categoría `MISSING/NOT SPECIFIED` concentra 911.250 registros, casi la mitad del dataset. Es una limitación de esta variable que conviene tener presente.

## Relación entre las variables

Para las numéricas uso correlación de Spearman en lugar de Pearson, porque `FIRE_SIZE` tiene una asimetría enorme (mediana de 1 acre, máximo de 606.945) y Spearman trabaja sobre rangos, así que no se deja llevar por los valores extremos.

Para las categóricas, así que uso V de Cramér, que mide asociación
entre una variable categórica y el objetivo en una escala de 0 a 1.

In [ ]:

d = df[df['SIN_CAUSA']==0].copy()
d['ES_NATURAL'] = (d['STAT_CAUSE_DESCR']=='Lightning').astype(int)
d['LOG_FIRE_SIZE'] = np.log1p(d['FIRE_SIZE'])

cols_num = ['FIRE_YEAR','DISCOVERY_DOY','LATITUDE','LONGITUDE','LOG_FIRE_SIZE','ES_NATURAL']
corr = d[cols_num].corr(method='spearman')

def cramers_v(x, y):
    t = pd.crosstab(x, y)
    chi2 = chi2_contingency(t)[0]
    n = t.values.sum()
    r, k = t.shape
    phi2 = max(0, chi2/n - ((k-1)*(r-1))/(n-1))
    rc = r - ((r-1)**2)/(n-1)
    kc = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2 / min(kc-1, rc-1))

v_cat = {c: cramers_v(d[c], d['ES_NATURAL'])
         for c in ['STATE','OWNER_DESCR','NWCG_REPORTING_AGENCY']}

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6), gridspec_kw={'width_ratios':[1.55, 1]})

ax = axes[0]
mask = np.triu(np.ones_like(corr, dtype=bool), k=0)
sns.heatmap(corr, mask=mask, cmap=DIV, vmin=-1, vmax=1, center=0,
            annot=True, fmt='.2f', annot_kws={'size':9},
            linewidths=1.2, linecolor='white', square=True,
            cbar_kws={'shrink':0.7, 'label':'Spearman'}, ax=ax)
ax.set_title('Variables numéricas', loc='left', pad=10)
ax.tick_params(axis='y', rotation=0)
ax.tick_params(axis='x', rotation=35)
ax.grid(visible=False)

ax = axes[1]
s = pd.Series(v_cat).sort_values()
ax.barh(range(len(s)), s.values, color=AZUL, height=0.5)
ax.set_yticks(range(len(s))); ax.set_yticklabels(s.index)
for i, v in enumerate(s.values):
    ax.text(v + 0.012, i, f'{v:.3f}', va='center', fontsize=9.5, color=TEXTO)
ax.set_xlim(0, s.max()*1.32)
ax.set_xlabel('V de Cramér contra la variable objetivo')
ax.set_title('Variables categóricas', loc='left', pad=10)
ax.grid(axis='y', visible=False)

fig.suptitle('Asociación entre las variables predictoras',
             x=0.005, ha='left', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
plt.savefig('../figuras/correlaciones.png')

Ninguna correlación individual con el objetivo pasa de 0,31 en valor
absoluto.

Que `DISCOVERY_DOY` dé apenas 0,21 y después sea la variable más importante del modelo (35,8%) es justamente el punto: Spearman solo detecta relaciones monótonas, y la del día del año no lo es, sube hasta julio y después baja. Un modelo de árboles sí captura esa forma.

En las categóricas los tres valores son parecidos (0,546, 0,541 y 0,514), pero hay que leerlos con cuidado: V de Cramér tiende a dar valores más altos cuando la variable tiene muchas categorías, y `STATE` tiene 52 mientras que la agencia tiene 11. No son directamente
comparables entre sí. (A revisar)